In [ ]:
# Papermill overrides these for each CSV row.
snp_calls_to_vcf_kwargs = {
    "output_path": "local_igv_vcfs/example.vcf.gz",
    "region": "2RL:2440000-2540000",
    "sample_sets": ["1354-VO-KE-DONNELLY-VMF00281"],
    "sample_query": "sample_id == 'VBS96796'",
}
result_csv = "snp_calls_to_vcf_benchmarks.csv"

In [ ]:
import csv
import json
import subprocess
import time
from pathlib import Path

import filelock
import malariagen_data
from memory_profiler import memory_usage

af1 = malariagen_data.Af1()
vcf_path = Path(snp_calls_to_vcf_kwargs["output_path"])
vcf_path.parent.mkdir(parents=True, exist_ok=True)

def export_vcf():
    try:
        af1.snp_calls_to_vcf(**snp_calls_to_vcf_kwargs)
    except ValueError: # if no relevant records
        pass

start_time = time.perf_counter()
peak_memory_mib = memory_usage((export_vcf, ()), max_usage=True)
runtime_sec = time.perf_counter() - start_time

snp_result = subprocess.run(
    ["bcftools", "view", "-H", "-v", "snps", str(vcf_path)],
    check=True,
    capture_output=True,
    text=True,
)
record = {
    "region": str(snp_calls_to_vcf_kwargs["region"]),
    "output_path": str(vcf_path),
    "sample_sets": json.dumps(snp_calls_to_vcf_kwargs["sample_sets"]),
    "sample_query": str(snp_calls_to_vcf_kwargs["sample_query"]),
    "SNPs_in_region": len(snp_result.stdout.splitlines()),
    "peak_memory_mib": round(peak_memory_mib, 2),
    "runtime_sec": round(runtime_sec, 4),
    "snp_calls_to_vcf_kwargs": json.dumps(snp_calls_to_vcf_kwargs, sort_keys=True),
}

result_path = Path(result_csv)
result_path.parent.mkdir(parents=True, exist_ok=True)
lock = filelock.FileLock(f"{result_path}.lock")
with lock:
    file_exists = result_path.exists() and result_path.stat().st_size > 0
    with result_path.open("a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=record.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(record)

print(f"Recorded: {record}")